# 🎬 WanGP — Free AI Video Studio (Google Colab)

Make **unlimited AI videos for free**. WanGP is open-source, so there are **no credits,
no limits, no subscription** — you're just borrowing Google's graphics card.

## Do these 2 things first

**1.** In the Colab menu click **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.**

**2.** Then click **Runtime → Run all** (or press ▶️ on each cell in order, top to bottom).

⏱️ First run takes **~10 minutes** (it downloads the AI software). Later runs are faster.

🆓 You get a free GPU for a few hours per day. If it ever disconnects, just Run all again.


---
### Cell 1 — Check that the free GPU is attached

In [ ]:
# Look for "Tesla T4" (or L4 / A100) and at least 15 GB of memory.
# If you see "No devices were found", go to Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi
!df -h /content | tail -1

---
### Cell 2 — Download WanGP (takes ~2 minutes)

In [ ]:
%cd /content
import os, shutil
if os.path.exists('/content/WanGP'):
    shutil.rmtree('/content/WanGP')
!git clone --depth 1 https://github.com/deepbeepmeep/WanGP.git
%cd /content/WanGP
print("Downloaded. Files:", len(os.listdir('/content/WanGP')))

---
### Cell 3 — Install the AI software (takes ~5 minutes, lots of scrolling text is normal)

In [ ]:
# This installs everything. If ONE optional package fails, do not panic -
# the next cell re-installs the important ones and the launcher will still work.
%cd /content/WanGP
!pip install -q -r requirements.txt

---
### Cell 4 — Safety net: make sure the important bits are really installed

In [ ]:
import importlib, subprocess, sys

REQUIRED = [
    "torch", "gradio", "diffusers", "transformers", "accelerate",
    "safetensors", "numpy", "imageio", "imageio_ffmpeg", "einops",
]

missing = []
for name in REQUIRED:
    try:
        importlib.import_module(name)
    except Exception:
        missing.append(name)

if missing:
    print("Re-installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All core packages are installed ✅")

import torch
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Graphics card   :", torch.cuda.get_device_name(0))
    print("Video memory    :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
else:
    print("!! NO GPU FOUND -> Runtime -> Change runtime type -> T4 GPU, then Run all again.")

---
### Cell 5 — Start the studio ▶️

Run this cell and **wait ~2 minutes**. When it finishes you will see a line like:

```
Running on public URL: https://ab12cd34ef.gradio.live
```

👉 **Click that `https://....gradio.live` link.** That is your WanGP studio.
Leave this cell running — if you stop it, the studio closes.

In [ ]:
import os, subprocess, sys, torch, re

# Pick the memory profile that matches the free GPU you were given.
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
else:
    vram = 0

if vram >= 38:
    profile = "5"          # A100 / L40S  - maximum quality
elif vram >= 22:
    profile = "5"          # A100 40GB / L4
elif vram >= 14:
    profile = "4"          # T4 16GB - the usual free Colab GPU
elif vram >= 10:
    profile = "3"
elif vram >= 6:
    profile = "2"
else:
    profile = "1"

print(f"Detected {vram:.1f} GB video memory -> using Profile {profile}")
print("Starting WanGP... look for the  https://....gradio.live  link below.\n")

# --share gives you a public link, --listen lets the dashboard accept connections.
!python wgp.py --share --listen --profile {profile}

---
# 🎉 Now make your video

1. Click the **`https://....gradio.live`** link from Cell 5.
2. In **Prompt**, describe your video in plain English, for example:
   > *a golden retriever surfing a wave at sunset, cinematic slow motion*
3. Leave everything else alone for your first video (the defaults are good).
4. Click **Generate**.
5. Wait a few minutes. Your video appears on the right — **click it and download**.

## If something goes wrong

| What you see | Fix |
|---|---|
| `CUDA out of memory` | Restart runtime and use a lower `--profile` (4 → 3 → 2), or pick a smaller model. |
| `No GPU / CUDA not available` | **Runtime → Change runtime type → T4 GPU**, then Run all again. |
| It's stuck downloading for ages | Normal on the first run — models are several GB. Wait it out once. |
| Runtime disconnected | Free GPUs sleep when idle. Just Run all again. |
| Video looks odd | Add detail to your prompt: subject + action + camera + lighting. |

## Want more?

- `VIDEO_GENERATION_GUIDE.md` in this repo — every model and setting explained
- `docs/` — the official WanGP documentation
